# 06 | Hyperparameter Tuning - RandomizedSearchCV y Optuna/TPE

## Objetivo del notebook

Este notebook documenta la optimización solicitada en Sprint 4. Se usan dos enfoques:

1. **RandomizedSearchCV:** exploración rápida de hiperparámetros.
2. **Optuna/TPE:** búsqueda bayesiana que aprende de trials anteriores.

La métrica de tuning es F2 porque da más peso al recall de la clase 1, coherente con el costo alto de falsos negativos.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


## 1. Por qué no usar GridSearch exhaustivo

GridSearch prueba todas las combinaciones. En datasets tabulares con pipelines, OHE y modelos pesados puede ser muy lento.

Por eso se prioriza:

- RandomizedSearchCV para explorar espacios grandes en pocas iteraciones.
- Optuna/TPE para búsqueda bayesiana más inteligente.

En `src/config.py` los valores están intencionalmente pequeños para que el proyecto pueda ejecutarse en laptop:

```python
RANDOM_SEARCH_ITER = 2
OPTUNA_TRIALS = 2
```

Para una entrega más robusta se pueden subir a 20, 50 o más trials si hay tiempo de cómputo.


In [ ]:
from src.tuning import default_random_search_spaces
spaces = default_random_search_spaces()
spaces.keys()


## 2. Modelos tuneados

Se tunean modelos del backlog, no modelos fuera del alcance:

| Modelo | Método | Por qué |
|---|---|---|
| Random Forest | RandomizedSearchCV | Es fuerte en tabular y sensible a profundidad/tamaño de hojas. |
| Decision Tree | RandomizedSearchCV | Controlar profundidad ayuda a reducir overfitting. |
| Logistic Regression | RandomizedSearchCV | Regularización `C` cambia el balance sesgo-varianza. |
| XGBoost | RandomizedSearchCV | Boosting suele ser competitivo en datos tabulares. |
| LightGBM | Optuna/TPE | Tiene varios hiperparámetros interactuando; Optuna es eficiente para esto. |


In [ ]:
# Ejecución oficial:
# PYTHONPATH=. python scripts/run_all.py
#
# El script guarda resultados en:
# reports/tuning_results.csv
# reports/optuna_lightgbm_trials.csv, si Optuna/LightGBM están instalados.


## 3. Métrica de tuning

Se usa F2 en vez de F1 porque el negocio penaliza más los FN que los FP.

- F1 pondera precision y recall igual.
- F2 da más peso a recall.
- Business value se calcula después, usando la matriz de confusión completa.

Esto permite que el proceso técnico esté alineado al problema financiero.


In [ ]:
from src.tuning import F2_SCORER
F2_SCORER


## 4. Interpretación esperada

Un modelo tuneado no siempre gana. Puede mejorar recall pero empeorar demasiados FP, o puede ser más complejo sin aportar valor económico.

Por eso el tuning se registra, pero la decisión final se toma en el notebook de Business Value.
